# Attention

Example:

> Buffalo buffalo Buffalo buffalo buffalo buffalo Buffalo buffalo.

The same token, `buffalo`, can mean different things depending on where it appears.

## 1. Goal

We want:

```text
sequence
↓
contextualized sequence
```

In other words, we want to turn each token's **initial representation**:

```text
x₁, x₂, x₃, ...
```

into a representation that incorporates context:

```text
h₁, h₂, h₃, ...
```

A useful intuition:

> Ambiguity lives in the initial representation; context resolves it.

---

## 2. Why do we need an initial representation?

If every token started with the same representation:

```text
I        → [0, 0, 0]
love     → [0, 0, 0]
buffalo  → [0, 0, 0]
pizza    → [0, 0, 0]
```

then:

```text
I love pizza
```

and:

```text
buffalo buffalo buffalo
```

would look identical to the model.

So contextualization must start from an **informative initial state**.

---

## 3. Why not one-hot?

One-hot vectors are already enough to distinguish token identity.

In fact:

```text
embedding lookup = one-hot × embedding matrix
```

So the question is not:

> Can we distinguish different tokens?

The real question is:

> What kind of starting representation is useful for later computation?

With one-hot vectors, all different tokens are:

```text
equally unrelated
```

An embedding instead gives each token a **learned initial representation**.

---

## 4. The real problem

Now we have:

```text
token
↓
initial embedding
```

The real question becomes:

> How do we turn a sequence of initial representations into contextualized representations?

Think of it as an API:

```python
contextualize(sequence) -> contextualized_sequence
```

---

## 5. Dumbest implementation: Giant MLP

Forget attention for now.

We could simply define:

```text
max_length = 1024
```

Pad shorter sequences and include positional information.

Then do:

```text
1024 embeddings
↓
flatten
↓
giant MLP
↓
1024 contextualized representations
```

In principle:

> This is a perfectly valid implementation of `contextualize()`.

So attention does not need to answer:

> "Can an MLP do this?"

It needs to answer:

> **Can we design a better contextualization operator than a Giant MLP?**

That is the real starting point for attention.

Q1：
“对于一个token，如何通过读取别的 token，让自己的 representation 发生了变化。”
MLP能做到吗？我们通过 UAT 知道它理论上能做。

一个token A，看到另一个token B，A要回答，站在 A 的角度，B 跟我相关吗？
那最开始的想法是什么？对比一下A和B，就是它们总要“交流”一下，对吧？

那目前MLP有吗？非常非常的不直接！

某些 hidden neuron 可以同时依赖 A 和 B，所以 mathematically 它们确实“交流”了。
但这个交流非常不 explicit。
你看不见一个东西叫：

relationship(A,B)

MLP 只是：把所有东西一起塞进去，让大量 weights 自己想办法把这种关系编码出来。


我们的问题本来是：A 怎么利用 B？

那最直接的方法当然不是：把 A、B、C、D 全塞进一个巨大函数，希望它自己悟出来。

而是：

先让 A 和 B 直接发生一次计算。

问题来了，怎么计算？


Q2:
great！那还是先用最蠢的想法，既然有了embedding vector，上cosine score啊！对吧，越像，越有用！

好的点：从从 sequence-level function
变成 token-to-token interaction。同一套规则可以拿去比较任意两个 token。它把复杂的两个 vector 之间的关系，压成了一个 scalar。

缺点：
1. cos(A,B)!=cos(B,A),cosine 这个函数结构上直接做不到
2. what A wants match what B has? 这个问题和“A和B像不像”其实很不一样

what A wants match what B has？
一个 embedding 里面可能包含很多东西：
当 A 判断 B 对自己有没有用的时候：
不应该默认 embedding 里的所有 information 都同等参与这个判断，或者说这些information具体怎么来帮助，贡献做这个判断，我们不知道
但我们真正可能想要：
从 A 里面抽出“我现在要找什么”
从 B 里面抽出“我这里有什么可以给你”
然后再比较这两个东西。


我们需要一个 score(A, B)。

它应该保留 cosine 的优点：
pairwise、reusable、representation-dependent、最后输出一个 scalar。
但是它还必须比 cosine 多做到几件事：
A 看 B 和 B 看 A 可以不同。

它衡量的是 compatibility / usefulness，而不只是 similarity。

它应该能决定 A 的哪些信息用于“寻找”，B 的哪些信息用于“被找到”。


Q3:
OK，因为我们要scalar -> dot乘法很有用！
对于一个token，它需要“代理”，不能自己亲自上
A 看 B 和 B 看 A 可以不同  -> 每一个token要有自己的query和key，这样queryA 查 keyB和queryB查keyA才能不同

所以就自然推出，一个token的embedding map到自己对应的query和key，然后算score（tokenA,tokenB)的时候，用queryA和keyB的dot product！

这是我自己根据我们目前的内容推出来的，没有看attention 公式作弊😂

真正打破 symmetry 的，不是“名字一个叫 query 一个叫 key”，而是它们是通过两个不同的 transformation 得到的。


Q4：如果 A 决定听 B，那 A 到底从 B 那里拿什么？Key 本身就是 B 真正要传给 A 的信息吗？
   